# Diabetic Retinopathy Grading
Train EfficientNet-B5 on the preprocessed dataset with stratified splits and weighted sampling.


In [1]:
import os

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
from sklearn.model_selection import train_test_split


/user/HS400/zv00033/miniconda3/envs/diab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Labels & Filter to Existing Images

In [ ]:
TRAIN_DIR = Path("dataset/train")
LABELS_CSV = Path("trainLabels.csv")
NUM_CLASSES = 5
IMG_SIZE = 456

df = pd.read_csv(LABELS_CSV)
# Build full file paths and keep only rows whose images actually exist
df["filepath"] = df["image"].apply(lambda x: str(TRAIN_DIR / f"{x}.jpeg"))
df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print(f"Total images found: {len(df)}")
print(f"\nClass distribution:\n{df['level'].value_counts().sort_index()}")

Total images found: 35126

Class distribution:
level
0    25810
1     2443
2     5292
3      873
4      708
Name: count, dtype: int64


## 2. Stratified Train / Val / Test Split (70 / 10 / 20)

In [4]:
# First split: 70% train, 30% temp (which becomes 20% test + 10% val)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["level"], random_state=42
)

# Second split: split the 30% into 20% test and 10% val  (2/3 and 1/3 of 30%)
test_df, val_df = train_test_split(
    temp_df, test_size=1/3, stratify=temp_df["level"], random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  Test: {len(test_df)}  Val: {len(val_df)}")
print(f"\nTrain class distribution:\n{train_df['level'].value_counts().sort_index()}")
print(f"\nTest class distribution:\n{test_df['level'].value_counts().sort_index()}")
print(f"\nVal class distribution:\n{val_df['level'].value_counts().sort_index()}")

Train: 24588  Test: 7025  Val: 3513

Train class distribution:
level
0    18067
1     1710
2     3704
3      611
4      496
Name: count, dtype: int64

Test class distribution:
level
0    5162
1     489
2    1058
3     175
4     141
Name: count, dtype: int64

Val class distribution:
level
0    2581
1     244
2     530
3      87
4      71
Name: count, dtype: int64


## 3. Class Weights for Loss-Based Balancing
Using cross entropy loss due to the class imbalance


In [5]:
class_counts = train_df["level"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize to sum to 1
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Using loss-based balancing (weighted CrossEntropyLoss).")
print(f"Class counts: {dict(enumerate(class_counts))}")
print(f"Normalized class weights: {dict(enumerate(class_weights.round(6)))}")


Using loss-based balancing (weighted CrossEntropyLoss).
Class counts: {0: np.int64(18067), 1: np.int64(1710), 2: np.int64(3704), 3: np.int64(611), 4: np.int64(496)}
Normalized class weights: {0: np.float64(0.01213), 1: np.float64(0.128163), 2: np.float64(0.059168), 3: np.float64(0.358688), 4: np.float64(0.441851)}


## 4. Custom Dataset & Data Augmentations

In [6]:
class DRDatasetFromPaths(Dataset):
    """Dataset that loads images from disk."""

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = int(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label


### Tranforming the datasets

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]       #ImageNet normalization for pretrained EfficientNet
IMAGENET_STD = [0.229, 0.224, 0.225]       #ImageNet normalization for pretrained EfficientNet

train_transform = T.Compose([
    T.RandomRotation(degrees=360),          #Allowing images to be rotated 360 degrees
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Build datasets -------------------------------------------------------
train_dataset = DRDatasetFromPaths(train_df["filepath"].tolist(), train_df["level"].tolist(), transform=train_transform)
test_dataset = DRDatasetFromPaths(test_df["filepath"].tolist(), test_df["level"].tolist(), transform=eval_transform)
val_dataset = DRDatasetFromPaths(val_df["filepath"].tolist(), val_df["level"].tolist(), transform=eval_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test  dataset size: {len(test_dataset)}")
print(f"Val   dataset size: {len(val_dataset)}")


Train dataset size: 24588
Test  dataset size: 7025
Val   dataset size: 3513


## 5. DataLoaders

Load the datasets into each variable - train_loader, val_loader, test_loader for training and testing.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## 6. Load EfficientNet-B5 from timm

In [15]:
model = timm.create_model("efficientnet_b5", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

print(model.default_cfg["input_size"])  #Expected input size of model
print(f"Classifier: {model.classifier}")    #Type of classifer model
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")  #Total parameters of model

(3, 448, 448)
Classifier: Linear(in_features=2048, out_features=5, bias=True)
Total parameters: 28,351,029


## 7. Training Configuration
Optimizer: AdamW. LR schedule: cosine annealing from `LR` to `1e-6` over all epochs. Mixed-precision scaler for AMP.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast

NUM_EPOCHS = 10
LR = 3e-4                                   #Starting with default best practices for AdamW
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 2                     #Stop training if QWK doesn't improve for 2 consecutive epochs
BEST_CKPT_PATH = "best_efficientnet_b5.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = GradScaler()

print(f"Optimizer : AdamW(lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"Schedule  : CosineAnnealingLR over {NUM_EPOCHS} epochs to eta_min=1e-6")
print(f"Batch     : {BATCH_SIZE}")
print(f"Epochs    : {NUM_EPOCHS} (early stopping patience={EARLY_STOP_PATIENCE} on val QWK)")

## 8. Train / Eval Helpers
`train_one_epoch` runs one epoch with AMP, gradient accumulation, and gradient clipping.
`evaluate` runs a no-grad pass and returns loss, accuracy, **quadratic weighted kappa** (the standard DR metric), and the confusion matrix.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from tqdm.auto import tqdm


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    return running_loss / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_samples = 0
    all_preds, all_labels = [], []

    for x, y in tqdm(loader, desc="eval", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with autocast():
            logits = model(x)
            loss = criterion(logits, y)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())

        running_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    return {
        "loss": running_loss / n_samples,
        "acc":  (y_pred == y_true).mean(),
        "qwk":  cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "cm":   confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))),
        "y_true": y_true,
        "y_pred": y_pred,
    }

## 9. Training Loop
Train, validate, step the LR scheduler, and save the best checkpoint by val QWK.
Stops early if val QWK does not improve for `EARLY_STOP_PATIENCE` epochs.

In [ ]:
best_qwk = -1.0
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_qwk": [], "lr": []}

for epoch in range(1, NUM_EPOCHS + 1):
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}  (lr={lr_now:.2e})")

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device,
    )
    val = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val["loss"])
    history["val_acc"].append(val["acc"])
    history["val_qwk"].append(val["qwk"])
    history["lr"].append(lr_now)

    print(f"  train_loss={train_loss:.4f}  val_loss={val['loss']:.4f}  val_acc={val['acc']:.4f}  val_qwk={val['qwk']:.4f}")

    if val["qwk"] > best_qwk:
        best_qwk = val["qwk"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_CKPT_PATH)
        print(f"  [best] new best val QWK: {best_qwk:.4f} (saved {BEST_CKPT_PATH})")
    else:
        epochs_no_improve += 1
        print(f"  no improvement ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val QWK: {best_qwk:.4f}")
            break

print(f"\nTraining done. Best val QWK: {best_qwk:.4f}")

## 10. Final Test Evaluation
Load the best checkpoint (by val QWK) and run it once on the held-out test set. Report QWK, accuracy, and the confusion matrix.

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device))
test = evaluate(model, test_loader, criterion, device)

print(f"Test loss : {test['loss']:.4f}")
print(f"Test acc  : {test['acc']:.4f}")
print(f"Test QWK  : {test['qwk']:.4f}")

print("\nConfusion matrix (rows=true, cols=pred):")
cm_df = pd.DataFrame(
    test["cm"],
    index=[f"true_{i}"  for i in range(NUM_CLASSES)],
    columns=[f"pred_{i}" for i in range(NUM_CLASSES)],
)
print(cm_df)